# Radar Tech Brasil - Análise Exploratória

Este notebook apresenta uma primeira leitura exploratória do mercado formal de tecnologia no Brasil a partir do Novo CAGED e da CBO.

A análise usa agregados processados pelo pipeline do projeto, evitando carregar os microdados completos em memória.

## 1. Contexto

O objetivo é transformar microdados públicos em indicadores analíticos sobre admissões, desligamentos, saldo, ocupações, categorias de tecnologia, distribuição geográfica, remuneração e perfil profissional.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

def read_agg(filename: str) -> pd.DataFrame:
    return pd.read_csv(PROCESSED_DIR / filename, sep=';')

## 2. Fonte dos Dados

- Novo CAGED: microdados públicos do Ministério do Trabalho e Emprego.
- CBO: Classificação Brasileira de Ocupações usada para mapear ocupações de tecnologia.
- Janela inicial: competências `202507` a `202606`.

A metodologia de classificação CBO tech está documentada em `docs/metodologia_cbo_tech.md`.

In [2]:
overview = read_agg('agg_tech_overview_mensal.csv')
category = read_agg('agg_tech_by_category_mensal.csv')
occupation = read_agg('agg_tech_by_occupation_mensal.csv')
uf = read_agg('agg_tech_by_uf_mensal_enriched.csv')
age = read_agg('agg_tech_by_age_group_mensal.csv')
education = read_agg('agg_tech_by_education_mensal_enriched.csv')

overview['competencia'] = overview['competencia'].astype(str)
category['competencia'] = category['competencia'].astype(str)
occupation['competencia'] = occupation['competencia'].astype(str)
uf['competencia'] = uf['competencia'].astype(str)
age['competencia'] = age['competencia'].astype(str)
education['competencia'] = education['competencia'].astype(str)

overview.head()

,competencia,total_registros_tech,total_admissoes,total_desligamentos,saldo_empregos,remuneracao_media,remuneracao_mediana,ocupacoes_analisadas,categorias_tech
0,202507,57760,29980,27780,2200,5358.06,2838.91,39,9
1,202508,58713,30364,28349,2015,5435.27,2986.55,39,9
2,202509,58042,30334,27708,2626,5486.39,2986.54,39,9
3,202510,60592,31000,29592,1408,5411.52,2900.00,39,9
4,202511,51324,27564,23760,3804,5430.84,2997.27,39,9


## 3. Qualidade dos Dados

O pipeline preserva registros e cria flags de qualidade em vez de remover dados automaticamente. Para remuneração média e mediana, foram considerados apenas salários maiores que zero e sem flag de salário extremo.

In [3]:
overview.describe(include='all')

,competencia,total_registros_tech,total_admissoes,total_desligamentos,saldo_empregos,remuneracao_media,remuneracao_mediana,ocupacoes_analisadas,categorias_tech
count,12,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.0,12.0
unique,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,202507,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,55723.833333,28393.250000,27330.583333,1062.666667,5411.461667,2887.789167,39.0,9.0
std,NaN,3714.625512,2698.877818,1500.806841,2296.580633,151.088995,124.040154,0.0,0.0
min,NaN,48027.000000,21341.000000,23760.000000,-5345.000000,5106.840000,2614.070000,39.0,9.0
25%,NaN,53540.000000,27456.000000,26659.750000,780.750000,5355.152500,2829.182500,39.0,9.0
50%,NaN,56593.000000,28971.500000,27364.000000,1648.000000,5433.055000,2923.880000,39.0,9.0
75%,NaN,58209.750000,30341.500000,28371.500000,2085.250000,5489.085000,2986.542500,39.0,9.0


## 4. Mercado Geral

A primeira leitura observa volume total, admissões, desligamentos e saldo mensal do recorte tech.

In [4]:
total_admissoes = int(overview['total_admissoes'].sum())
total_desligamentos = int(overview['total_desligamentos'].sum())
saldo = int(overview['saldo_empregos'].sum())

pd.DataFrame([
    {'metrica': 'Admissões tech', 'valor': total_admissoes},
    {'metrica': 'Desligamentos tech', 'valor': total_desligamentos},
    {'metrica': 'Saldo tech', 'valor': saldo},
])

,metrica,valor
0,Admissões tech,340719
1,Desligamentos tech,327967
2,Saldo tech,12752


## 5. Evolução Temporal

A série mensal evita resumir a janela em um único número e mostra meses com comportamento distinto.

In [5]:
px.line(
    overview,
    x='competencia',
    y=['total_admissoes', 'total_desligamentos', 'saldo_empregos'],
    markers=True,
    labels={'value': 'Registros', 'competencia': 'Competência', 'variable': 'Métrica'},
    title='Evolução mensal de admissões, desligamentos e saldo tech',
)

## 6. Categorias de Tecnologia

Categorias ajudam a separar ocupações de desenvolvimento, suporte, redes, infraestrutura, dados, segurança e gestão.

In [6]:
category_total = (
    category.groupby('categoria_tech', as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
)

px.bar(
    category_total,
    x='registros',
    y='categoria_tech',
    orientation='h',
    labels={'registros': 'Registros', 'categoria_tech': 'Categoria'},
    title='Volume por categoria tech',
)

## 7. Ocupações

O ranking de ocupações mostra onde está concentrado o volume de movimentações formais no recorte CBO tech.

In [7]:
occupation_total = (
    occupation.groupby(['codigo_cbo', 'ocupacao', 'categoria_tech'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
    .head(15)
)

occupation_total

,codigo_cbo,ocupacao,categoria_tech,registros,saldo_empregos
18,212405,Analista de desenvolvimento de sistemas,Desenvolvimento de Software,138835,4607
31,317210,Técnico de suporte ao usuário de tecnologia da...,Suporte Tecnico,71007,6825
21,212420,Analista de suporte computacional,Suporte Tecnico,66108,2422
29,317110,Desenvolvedor de sistemas de tecnologia da inf...,Desenvolvimento de Software,59327,635
38,732130,Instalador-reparador de redes telefônicas e de...,Redes,56291,3643
26,313220,Técnico em manutenção de equipamentos de infor...,Suporte Tecnico,30549,-1263
19,212410,Analista de redes e de comunicação de dados,Redes,27601,-635
28,313310,Técnico de rede (telecomunicações),Redes,23713,-109
34,731320,Instalador-reparador de linhas e aparelhos de ...,Redes,22112,-2560
30,317205,Operador de computador,Infraestrutura,21658,368


## 8. Estados

A análise geográfica usa UF enriquecida com sigla, nome e região.

In [8]:
uf_total = (
    uf.groupby(['uf', 'uf_sigla', 'uf_nome', 'regiao_nome'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('registros', ascending=False)
)

px.bar(
    uf_total.head(20),
    x='uf_sigla',
    y='saldo_empregos',
    labels={'uf_sigla': 'UF', 'saldo_empregos': 'Saldo'},
    title='Saldo tech por UF',
)

## 9. Salários

A remuneração é analisada com a regra documentada de salários válidos. A mediana é especialmente útil porque reduz a influência de valores extremos.

In [9]:
px.line(
    overview,
    x='competencia',
    y=['remuneracao_media', 'remuneracao_mediana'],
    markers=True,
    labels={'value': 'Remuneração', 'competencia': 'Competência', 'variable': 'Métrica'},
    title='Evolução de remuneração média e mediana',
)

## 10. Perfil Profissional

Faixa etária e escolaridade ajudam a entender o perfil dos vínculos formais no recorte tech.

In [10]:
age_total = (
    age.groupby('faixa_etaria', as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
)
age_order = ['Ate 20', '21-25', '26-30', '31-35', '36-40', '41-50', '51+', 'Nao informado']
age_total['ordem'] = age_total['faixa_etaria'].apply(lambda value: age_order.index(value) if value in age_order else 99)
age_total = age_total.sort_values('ordem').drop(columns='ordem')

px.bar(age_total, x='faixa_etaria', y='registros', title='Registros por faixa etária')

In [11]:
education_total = (
    education.groupby(['grau_instrucao', 'escolaridade'], as_index=False)
    .agg(registros=('registros', 'sum'), saldo_empregos=('saldo_empregos', 'sum'))
    .sort_values('grau_instrucao')
)

education_total

,grau_instrucao,escolaridade,registros,saldo_empregos
0,1,Analfabeto,699,-181
1,2,Fundamental incompleto,663,-37
2,3,Fundamental completo,533,-129
3,4,Medio incompleto,2793,-265
4,5,Medio completo,7218,-198
5,6,Superior incompleto,15186,468
6,7,Superior completo,265647,403
7,8,Mestrado,99453,10617
8,9,Doutorado,220494,1202
9,10,Pos-graduacao completa,5985,55


## 11. Principais Insights

Os insights gerados automaticamente estão documentados em `docs/insights_iniciais.md`.

Eles devem ser lidos como evidências descritivas, não como inferências causais.

## 12. Limitações

- O recorte depende do mapeamento CBO tech `v0.1`.
- CBO não captura senioridade, stack, modalidade remota ou tipo de contrato em detalhe moderno.
- A janela inicial possui 12 competências.
- RAIS, IBGE e Banco Central ainda não foram integrados.

## 13. Próximas Análises

- Salário por categoria e UF.
- Sazonalidade mensal.
- Comparação entre regiões.
- Revisão do mapeamento CBO tech com novas fontes e validação qualitativa.
- Carga PostgreSQL detalhada para consultas mais flexíveis.